# AI-Based Constrained Payment Routing Optimization

**Final Project Notebook — hybrid research/product presentation**

The project follows one design principle:

> **Add complexity only when the simpler formulation fails.**

Route averages → contextual prediction → temporal memory → continuous decay → stage-specific models → constrained optimization → online shadow pricing → prospective validation.

## 0. Colab bootstrap — clone the GitHub repository

In [ ]:
REPO_URL = "https://github.com/orankedem/flexfactor-routing-final-project"

from pathlib import Path
import os, sys, subprocess

LOCAL_REPO = Path("/content/flexfactor-routing-final-project")

if not (LOCAL_REPO / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(LOCAL_REPO)], check=True)

if (LOCAL_REPO / "src").exists():
    os.chdir(LOCAL_REPO)
    if str(LOCAL_REPO) not in sys.path:
        sys.path.insert(0, str(LOCAL_REPO))
    print("Repository ready:", LOCAL_REPO)

## 1. Install dependencies

In [ ]:
!pip -q install -r requirements.txt

## 2. Mount Google Drive and point to the raw data

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/FlexFactor_Final_Project")
DATA_PATH = DRIVE_PROJECT_ROOT / "data" / "transactions.parquet"
DRIVE_ARTIFACT_ROOT = DRIVE_PROJECT_ROOT / "artifacts"
DRIVE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

# False = fast presentation/reference mode.
# True = load and work from the real Drive dataset.
REBUILD_FROM_RAW = False

print("DATA_PATH:", DATA_PATH)
print("Exists:", DATA_PATH.exists())

## 3. Imports

The notebook is the **story/orchestration layer**. The reusable implementation is in `src/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_table, standardize_schema, basic_audit, attach_prior_attempt_state
from src.features import add_static_features, add_cross_features, build_multiscale_history
from src.models import compare_model_families
from src.optimization import solve_assignment_lp, simulate_pressure_policy, simulate_greedy_policy, capacity_audit
from config import STATIC_CROSS_SPECS, HISTORY_GROUPS, HALF_LIVES_DAYS, PRIOR_STRENGTH, CAPACITY_SHIFT

# Part I — Financial decision problem

For event $i$, candidate route $r$ and time $t$:

$$
\hat p_{ir,t}=P(	ext{success}\mid X_i,r,H_t,	ext{prior attempt state})
$$

The objective is to choose a feasible route online. A2/A3 are conditioned on previous failure and therefore have additional information that A1 does not have.

## 4. Load and standardize the Drive data

In [ ]:
COLUMN_OVERRIDES = {
    # Example:
    # "timestamp": "YourRawTimestampColumn",
    # "success": "YourRawTargetColumn",
}

if REBUILD_FROM_RAW:
    raw = load_table(DATA_PATH)
    attempts = standardize_schema(raw, overrides=COLUMN_OVERRIDES, verbose=True)
    display(basic_audit(attempts))
else:
    print("Fast mode: raw Drive data is not loaded. Set REBUILD_FROM_RAW=True for the real pipeline.")

# Part II — EDA: route signal and concept drift

The first questions were:

1. Does route choice contain predictive signal after basic traffic-mix adjustment?
2. Is the environment stationary?

Chronological analysis showed meaningful approval-rate drift, and the first model underpredicted later traffic. Therefore the model needed a representation of **current state**, not only static identities.

## 5. Chronological drift visualization

In [ ]:
if REBUILD_FROM_RAW:
    a1 = attempts[attempts["attempt"] == 1].copy()
    a1["month"] = a1["timestamp"].dt.to_period("M").astype(str)
    monthly = a1.groupby("month", as_index=False).agg(success_rate=("success","mean"), rows=("success","size"))
    display(monthly)

    fig, ax = plt.subplots(figsize=(9,4))
    ax.plot(monthly["month"], monthly["success_rate"], marker="o")
    ax.set_title("A1 success rate over time — evidence of non-stationarity")
    ax.set_xlabel("Month"); ax.set_ylabel("Observed success rate")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()
else:
    print("Reference development example: train ~7.17%, calibration ~7.28%, later test ~8.91%.")

# Part III — Model-family progression

The project did not start by assuming XGBoost.

- **LightGBM** — first feasibility baseline; good ranking, but exposed calibration drift.
- **CatBoost** — extensive categorical/history ablations and cold-start experiments.
- **XGBoost** — final frozen family used for A1/A2/A3 and final policy evaluation.

The next cell is an optional **fresh same-split/same-feature** comparison. It is separate from the historical progression so we do not pretend that experiments from different phases were apples-to-apples.

## 6. Optional controlled LightGBM vs CatBoost vs XGBoost comparison

In [ ]:
RUN_FRESH_FAMILY_COMPARISON = False

if REBUILD_FROM_RAW and RUN_FRESH_FAMILY_COMPARISON:
    a1 = attempts[attempts["attempt"] == 1].copy()
    a1 = add_static_features(a1)
    a1 = add_cross_features(a1, STATIC_CROSS_SPECS)

    feature_cols = [c for c in [
        "merchant_id","issuer_name","card_network","card_type","card_level","bank_category","mcc","route",
        "amount_numeric","amount_log1p","hour_sin","hour_cos","dow_sin","dow_cos","is_weekend",
        "bankcat_route","cardnetwork_route","cardtype_route","cardlevel_route","mcc_route"
    ] if c in a1.columns]

    train = a1[a1["timestamp"] < "2026-03-01"].copy()
    valid = a1[(a1["timestamp"] >= "2026-03-01") & (a1["timestamp"] < "2026-05-01")].copy()

    family_results, family_models = compare_model_families(train, valid, feature_cols)
    display(family_results)
else:
    print("Fresh family comparison disabled; the notebook preserves the real historical model progression.")

# Part IV — From fixed windows to continuous temporal memory

The first temporal solution summarized history over fixed windows so a tabular model could retain memory without requiring an LSTM/RNN.

A hard window has an artificial cliff: 29.9-day-old information can count fully while 30.1-day-old information disappears.

That motivated continuous decay:

$$
w(\Delta t)=0.5^{\Delta t/h}$$

Multiple half-lives represent different speeds of change.

## 7. Tiny decay example

In [ ]:
example = pd.DataFrame({"age_days":[0,3,6,9]})
example["weight_h3"] = 0.5 ** (example["age_days"] / 3)
display(example)

## 8. Build temporal state

The implementation stores only weighted successes, weighted count and last timestamp for each group. It does **not** rescan all previous transactions for every event.

Historical groups include merchant, issuer, route and contextual interactions such as merchant×route and issuer×route.

In [ ]:
BUILD_TEMPORAL_FEATURES = False

if REBUILD_FROM_RAW and BUILD_TEMPORAL_FEATURES:
    attempts2 = attach_prior_attempt_state(attempts)
    attempts2 = add_static_features(attempts2)
    attempts2 = add_cross_features(attempts2, STATIC_CROSS_SPECS)

    a1 = attempts2[attempts2["attempt"] == 1].copy()
    a1_features = build_multiscale_history(
        a1,
        HISTORY_GROUPS,
        half_lives_days=HALF_LIVES_DAYS,
        prior_rate=0.08,
        prior_strength=PRIOR_STRENGTH,
        verbose=True,
    )

    checkpoint = DRIVE_ARTIFACT_ROOT / "a1_multiscale_features.parquet"
    a1_features.to_parquet(checkpoint, index=False)
    print("Saved heavy checkpoint:", checkpoint)
else:
    print("Temporal rebuild is disabled by default because it is the heaviest stage. Full implementation: src/features.py")

# Part V — Stage-specific models

$$P(S_1\mid X,r_1,H_t)$$

$$P(S_2\mid X,r_2,A1_{observed},H_t)$$

$$P(S_3\mid X,r_3,A1_{observed},A2_{observed},H_t)$$

Current-attempt response codes are leakage. A2 may use A1 response state; A3 may use A1/A2 state.

## 9. Final factual predictive metrics

In [ ]:
predictive = pd.read_csv("artifacts/reference_predictive_metrics.csv")
display(predictive)

These are **observed-route factual metrics**. Average Precision is especially relevant because success becomes rare in later attempts; log loss/Brier/ECE matter because the optimizer uses the probabilities themselves.

# Part VI — From prediction to constrained optimization

With unlimited route capacity:

$$r_i^*=rg\max_r \hat p_{ir}$$

But under the working ±30% route-volume constraint, transactions compete for scarce route capacity.

## 10. Offline LP oracle and the meaning of “100% opportunity”

Success objective:

$$\max_x \sum_{i,r}x_{ir}\hat p_{ir}$$

Approved-value objective:

$$\max_x \sum_{i,r}x_{ir}Amount_i\hat p_{ir}$$

The LP sees the entire period, so it is a **full-hindsight model-based oracle**.

**100% opportunity means:** the maximum incremental model-implied objective found by that LP relative to the model-scored historical routing baseline under the same constraints.

- Success LP: **+188.8 expected approvals**, **+1.299M expected approved transaction value**
- Value LP: **+173.1 expected approvals**, **+1.424M expected approved transaction value**

Following discussion with the company data scientist, policy comparison is intentionally based on calibrated model probabilities. This makes the offline comparison internally realistic under the predictive model; it does not make counterfactual outcomes observed.

## 11. Reference policy results

In [ ]:
policy_results = pd.read_csv("artifacts/reference_policy_results.csv")
display(policy_results)

# Part VII — Greedy failure → pressure / shadow pricing

Greedy selects the locally best route but does not price the future opportunity cost of consuming scarce route capacity now.

The pressure policy uses:

$$
Pressure_{r,t}=rac{A_{r,t}-B_{r,t}}{\max(0.3B_{r,t},1)}
$$

and:

$$
Score_{ir,t}=\hat p_{ir,t}-\lambda Pressure_{r,t}.
$$

The adjusted score chooses the route. **Expected approvals are still evaluated using the raw probability $\hat p$, not the pressure-adjusted score.**

## 12. Opportunity-capture visualization

In [ ]:
plot_df = policy_results[policy_results["policy"].isin(["greedy","success_pressure","LP_success"])].copy()
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(plot_df["policy"], 100*plot_df["success_opportunity_capture"])
ax.set_ylabel("Share of LP success opportunity (%)")
ax.set_title("Online routing vs full-hindsight LP oracle")
ax.set_ylim(0,105)
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

# Part VIII — Scientific limitation and next validation step

Historical data observes only:

$$Y_i(R_{logged})$$

not:

$$Y_i(R_{alternative}).$$

Therefore:

- predictive metrics on logged routes are **factual**;
- route movement/capacity statistics are **operational**;
- expected alternative-route approvals/value are **model-implied**;
- realized causal uplift requires a prospective controlled online / A-B test.

## 13. Final validation status

In [ ]:
status = pd.DataFrame([
    ["Predictive model quality", "Chronologically validated on observed routes"],
    ["Capacity feasibility", "Validated in policy replay"],
    ["LP / pressure policy gain", "Model-implied counterfactual estimate"],
    ["True causal production lift", "Not yet validated — requires online A/B test"],
], columns=["component","status"])
display(status)

# Final conclusion

The final system is not merely an approval classifier. It combines **dynamic temporal state**, **stage-specific probability estimation**, and **constrained online allocation**.

Complexity was introduced only where a simpler formulation failed:

concept drift → memory → continuous decay → contextual state → A1/A2/A3 → scarcity → LP benchmark → online pressure → prospective validation.